# Event Streaming — Learned One Step at a Time
### VidyaSankalp — Agent-AI Running Notes

We build **one example** — a Goa trip-planning agent — and grow it step by step. Each step adds exactly one new idea, and nothing is introduced until the previous step has landed.

We use only **one API** throughout: `agent.stream_events(input, version="v3")`.

**The steps:**
1. Watch the answer stream in, live — `message.text`
2. Watch the model's private thinking — `message.reasoning`
3. Watch the model *write* a tool call — `message.tool_calls`
4. Watch a tool actually *run* — `stream.tool_calls`
5. Watch the whole agent's state, and its final result — `stream.values` / `stream.output`
6. Put it all together: a budget that forces the agent to plan, execute, and re-plan
7. Let a tool narrate its own progress — `get_stream_writer()` + a custom transformer

Let's start.


## Setup


In [ ]:
%pip install -q langchain langchain-anthropic langchain-openai


In [17]:
import os

# Set ONE of these depending on the provider you're using.
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["OPENAI_API_KEY"] = "sk-..."

MODEL_NAME = "gpt-5"   # or e.g. "openai:gpt-4o-mini"


In [18]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

True

## Step 1 — Just watch the answer stream in

The smallest possible agent: **one tool**, `check_weather`. We ask a question and watch the answer type out live, instead of waiting for the whole thing.

The only piece of the API we need: `stream.messages`, and on each message, `.text`.


In [19]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


def check_weather(city: str) -> str:
    """Get the current weather forecast for a city."""
    forecasts = {
        "goa": "Sunny, 32°C, light humidity — good beach weather.",
        "hyderabad": "Partly cloudy, 29°C.",
    }
    return forecasts.get(city.strip().lower(), f"Weather data not available for {city}.")


agent = create_agent(
    model=init_chat_model(MODEL_NAME),
    tools=[check_weather],
)

print("Agent ready with 1 tool.")


Agent ready with 1 tool.


In [20]:
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "What's the weather in Goa?"}]},
    version="v3",
)

# stream.messages -> each LLM reply, as it's generated
# message.text    -> that reply's text, delta by delta (one small chunk at a time)
for message in stream.messages:
    for delta in message.text:
        print(delta, end="", flush=True)


Goa weather: Sunny, around 32°C with light humidity — good beach weather.

That's the core idea of Event Streaming: instead of `agent.invoke(...)` making you wait for the entire answer, `stream_events(...)` hands it to you as it's being written.


## Step 2 — Watch the model's private thinking

Before some models write their final answer, they work through the problem in a separate, private stream — their **reasoning**. It's not the answer itself; it's the model's own scratch-pad ("the weather in Goa should be sunny this time of year...").

New piece of the API: `message.reasoning`. It streams exactly like `message.text` — delta by delta — just on a different channel.

> Not every model exposes this. If yours doesn't, this loop simply prints nothing and moves on — that's expected, not an error.


In [21]:
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "What's the weather in Goa, and should I pack an umbrella?"}]},
    version="v3",
)

for message in stream.messages:
    for delta in message.reasoning:
        print(f"[thinking] {delta}", end="", flush=True)
    for delta in message.text:
        print(delta, end="", flush=True)


**`message.reasoning` vs `message.text`, plainly:** `.reasoning` is the model figuring things out *before* committing to words; `.text` is what it actually says to the user. Same mechanism (streamed deltas), two different channels on the same message.


## Step 3 — Watch the model *write* a tool call

Now we add a second tool, `search_flights`, and ask a question that needs it. Before the tool runs, the model has to decide to call it — and it writes out the tool's name and arguments, token by token, the same way it writes its reply text.

New piece of the API: `message.tool_calls`. Still on the `"messages"` feed — this is the model's side of a tool call, not the tool's side. The tool hasn't run yet at this point.


In [22]:
def search_flights(origin: str, destination: str) -> dict:
    """Search for a one-way flight and its price."""
    return {
        "origin": origin,
        "destination": destination,
        "one_way_price_inr": 9000,
        "duration": "2h 10m",
    }


agent = create_agent(
    model=init_chat_model(MODEL_NAME),
    tools=[check_weather, search_flights],
)

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "Find me a flight from Hyderabad to Goa, and tell me the price."}]},
    version="v3",
)

for message in stream.messages:
    for chunk in message.tool_calls:
        print(f"[model is writing a tool call] {chunk}")

    finalized = message.tool_calls.get()   # the completed call, once the model is done writing it
    if finalized:
        print("[finalized tool call]", finalized)

    for delta in message.text:
        print(delta, end="", flush=True)


[model is writing a tool call] {'type': 'tool_call_chunk', 'id': 'call_U4tJjaFLrjO3SZLJPayo5PNx', 'name': 'search_flights', 'args': '', 'index': 0}
[model is writing a tool call] {'type': 'tool_call_chunk', 'id': 'call_U4tJjaFLrjO3SZLJPayo5PNx', 'name': 'search_flights', 'args': '{"origin', 'index': 0}
[model is writing a tool call] {'type': 'tool_call_chunk', 'id': 'call_U4tJjaFLrjO3SZLJPayo5PNx', 'name': 'search_flights', 'args': '{"origin', 'index': 0}
[model is writing a tool call] {'type': 'tool_call_chunk', 'id': 'call_U4tJjaFLrjO3SZLJPayo5PNx', 'name': 'search_flights', 'args': '{"origin":"Hy', 'index': 0}
[model is writing a tool call] {'type': 'tool_call_chunk', 'id': 'call_U4tJjaFLrjO3SZLJPayo5PNx', 'name': 'search_flights', 'args': '{"origin":"Hy', 'index': 0}
[model is writing a tool call] {'type': 'tool_call_chunk', 'id': 'call_U4tJjaFLrjO3SZLJPayo5PNx', 'name': 'search_flights', 'args': '{"origin":"Hyderabad","', 'index': 0}
[model is writing a tool call] {'type': 'tool_c

**Where we are so far, on the `"messages"` feed alone:** `.reasoning` (private thinking), `.text` (what it says), and now `.tool_calls` (what it's asking to run, and with what arguments) — three different channels on the same stream of messages. Notice: nothing has actually *executed* yet. That's next.


## Step 4 — Watch the tool actually *run*

Section 3 showed the model deciding to call `search_flights`. Now let's watch the tool itself execute — a separate moment in time, after the model is done writing the call.

New piece of the API: `stream.tool_calls`. This is a *different feed* from `message.tool_calls` — it lives directly on `stream`, not inside a message, because it's about the tool running, not about what the model said.


In [29]:
# stream = agent.stream_events(
#     {"messages": [{"role": "user", "content": "Find me a flight from Hyderabad to Goa, and tell me the price."}]},
#     version="v3",
# )
#
# for message in stream.messages:
#     for delta in message.text:
#         print(delta, end="", flush=True)


Here’s what I found:
- Route: Hyderabad (HYD) to Goa (GOI)
- One-way price: ₹9,000
- Approximate duration: 2h 10m

Prices can vary by date and airline. Want me to check specific travel dates or times?

**The two tool-call moments, side by side, now that you've seen both:**
- `message.tool_calls` (Step 3) = the model **deciding** what to call and **writing the arguments** — nothing has run.
- `stream.tool_calls` (Step 4) = the tool **actually running**, with a real return value or an error.


## Step 5 — Watch the whole agent's state, and its final result

Everything so far has been about *messages* (what's being said or called) — one piece at a time. Sometimes you want the bigger picture: the entire conversation so far, as one snapshot.

Two new pieces of the API:
- `stream.values` — a snapshot of the **whole state**, after every step. Not one message — everything accumulated so far.
- `stream.output` — the **final** state, only available once the whole run is done.


In [ ]:
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "Find me a flight from Hyderabad to Goa, and tell me the price."}]},
    version="v3",
)

for snapshot in stream.values:
    print("state snapshot -> messages so far:", len(snapshot["messages"]))

final_state = stream.output
print("\nFinal state -> total messages:", len(final_state["messages"]))
print("Final answer:", final_state["messages"][-1].text)


**`stream.values` vs `stream.output`:** `.values` streams live, step by step, while the run is happening. `.output` is only available *after* the loop — one snapshot, the finished result. You've now seen every major feed `stream_events` offers: `messages` (with `.text`, `.reasoning`, `.tool_calls`), `tool_calls`, `values`, and `output`.


## Step 6 — Put it all together: a budget that forces planning

Now the real example. A third tool, `estimate_total_cost`, and a **system prompt** telling the agent to:

1. Announce a short plan before doing anything.
2. Execute it by calling tools.
3. Check the total cost against the user's budget.
4. If it's over budget, say so, revise the plan, and try again.
5. Give one final recommendation.

**Re-planning isn't a new streaming feature.** It's just more `message.text`, driven by the system prompt. Everything we need to watch it, we already have from Steps 1–5.


In [31]:
def estimate_total_cost(flight_one_way_price_inr: float, nights: int, hotel_per_night_inr: float = 3000) -> dict:
    """Estimate total trip cost: round-trip flight + hotel for the given nights."""
    flight_total = flight_one_way_price_inr * 2
    hotel_total = nights * hotel_per_night_inr
    total = flight_total + hotel_total
    return {
        "flight_total_inr": flight_total,
        "hotel_total_inr": hotel_total,
        "total_inr": total,
    }


SYSTEM_PROMPT = """You are a careful trip-planning assistant.

For every request, follow this loop and show your work in the chat:
1. PLAN: Write a short numbered plan (2-4 steps) before calling any tool.
2. EXECUTE: Call tools one at a time to carry out the plan.
3. CHECK: Once you know the total cost, compare it against the user's stated budget.
4. RE-PLAN IF NEEDED: If the total is over budget, say "Revising the plan:" followed by
   what you're changing, then execute the revised plan.
5. RECOMMEND: Give one final, clear recommendation.

The flight price you get back is fixed - if the trip is over budget, reduce the number
of nights instead, and call estimate_total_cost again with the new number.
"""

agent = create_agent(
    model=init_chat_model(MODEL_NAME),
    tools=[check_weather, search_flights, estimate_total_cost],
    system_prompt=SYSTEM_PROMPT,
)

print("Agent ready with 3 tools and a plan-execute-replan system prompt.")


Agent ready with 3 tools and a plan-execute-replan system prompt.


Now we watch it live, combining the feeds from Steps 1–4 with `stream.interleave(...)`, so everything prints in the order it actually happens — including `.reasoning` from Step 2, if your model produces any.


In [35]:
user_request = (
    "Plan a 5-night trip to Goa from Hyderabad. My budget is INR 25,000 total. "
    "Also tell me what the weather will be like."
)

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": user_request}]},
    version="v3",
)

for name, item in stream.interleave("messages", "tool_calls"):
    if name == "messages":
        for delta in item.text:
            print(delta, end="", flush=True)
        else:
            print("/n")
            print("Text completed----------")
        # item.reasoning -> the model's private thinking, before it commits to words (Step 2)
        for delta in item.reasoning:
            print(f"[thinking] {delta}", end="", flush=True)

        # item.tool_calls -> the model writing a call (Step 3)
        for chunk in item.tool_calls:
            print(f"\n[writing tool call] {chunk}", end="", flush=True)
        else:
            print("/n")
            print("tool calls completed----------")
        # item.text -> the plan, the re-plan, the final recommendation (Step 1)

    elif name == "tool_calls":
        # a tool actually executing (Step 4)
        print(f"\n[tool ran] {item.tool_name}({item.input}) -> {item.output}")

print("\n\n--- run finished ---")


PLAN:
1) Search one-way flight HYD -> Goa and check Goa weather in parallel.
2) Estimate total trip cost for 5 nights using the flight price.
3) Compare total to INR 25,000 budget.
4) If over budget, reduce nights and re-estimate until it fits; then recommend.

EXECUTE: Running flight search and weather check in parallel./n
Text completed----------
/n
reasoning completed----------

[writing tool call] {'type': 'tool_call_chunk', 'id': 'call_iYxV2zeZXpbSZsQFtJfkMb7m', 'name': 'search_flights', 'args': '{"origin": "Hyderabad", "desti', 'index': 0}
[writing tool call] {'type': 'tool_call_chunk', 'id': 'call_iYxV2zeZXpbSZsQFtJfkMb7m', 'name': 'search_flights', 'args': '{"origin": "Hyderabad", "desti', 'index': 0}
[writing tool call] {'type': 'tool_call_chunk', 'id': 'call_iYxV2zeZXpbSZsQFtJfkMb7m', 'name': 'search_flights', 'args': '{"origin": "Hyderabad", "desti', 'index': 0}
[writing tool call] {'type': 'tool_call_chunk', 'id': 'call_iYxV2zeZXpbSZsQFtJfkMb7m', 'name': 'search_flights', '

**What to look for:**
- `[thinking]` lines, if your model exposes reasoning, before it commits to the plan.
- A numbered **plan**, as plain text, before any tool runs.
- `search_flights` and `estimate_total_cost` running, with real numbers.
- If the first total is over budget: *"Revising the plan: reducing to 3 nights"* — plain text again — followed by a **second** `estimate_total_cost` call.
- A final recommendation once the numbers fit.


In [ ]:
# Now inspect the finished run with stream.output (Step 5), the way you'd check
# afterward how many planning attempts it took.
final_state = stream.output

tool_result_count = 0
for msg in final_state["messages"]:
    if getattr(msg, "type", None) == "tool":
        tool_result_count += 1

print("Total tool calls made this run:", tool_result_count)
print("(3 tool calls = fit the budget on the first try;")
print(" 4+ tool calls = it had to re-plan and search again.)")


## Step 7 — Let a tool speak for itself, live

Everything so far came from the model (`message.text`, `message.reasoning`, `message.tool_calls`) or from watching a tool's start/finish (`stream.tool_calls`). There's one more source of live updates: **the tool's own code**, narrating its progress while it runs — "Searching flights...", "Got a price" — independent of what it eventually returns.

New piece: `get_stream_writer()`, called *inside* the tool. It's not a `stream_events` projection by default — you have to register a small **transformer** to expose it as one. Treat the transformer class below as a recipe to copy, not something to memorize.


In [ ]:
from langgraph.config import get_stream_writer
from langgraph.stream import ProtocolEvent, StreamChannel, StreamTransformer


def check_weather_verbose(city: str) -> str:
    """Get the current weather forecast for a city, narrating its own progress."""
    writer = get_stream_writer()
    writer(f"Looking up weather for {city}...")

    forecasts = {
        "goa": "Sunny, 32°C, light humidity — good beach weather.",
        "hyderabad": "Partly cloudy, 29°C.",
    }
    result = forecasts.get(city.strip().lower(), f"Weather data not available for {city}.")

    writer(f"Done looking up {city}.")
    return result


# This class watches the raw event stream for "custom" events (the writer() calls above)
# and exposes them as a new projection: stream.extensions["custom"].
class ToolProgressTransformer(StreamTransformer):
    required_stream_modes = ("custom",)

    def __init__(self, scope: tuple[str, ...] = ()) -> None:
        super().__init__(scope)
        self.log = StreamChannel()

    def init(self) -> dict:
        return {"custom": self.log}

    def process(self, event: ProtocolEvent) -> bool:
        if event["method"] == "custom":
            self.log.push(event["params"]["data"])
        return True


agent = create_agent(
    model=init_chat_model(MODEL_NAME),
    tools=[check_weather_verbose],
)

print("Agent ready, with a tool that narrates its own progress.")


In [ ]:
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "What's the weather in Goa?"}]},
    version="v3",
    transformers=[ToolProgressTransformer],
)

# IMPORTANT: read both feeds TOGETHER, not one after the other.
# stream.interleave(...) accepts your custom projection key ("custom") by name,
# right alongside the built-in ones like "messages".
for name, item in stream.interleave("messages", "custom"):
    if name == "messages":
        for delta in item.text:
            print(delta, end="", flush=True)
    elif name == "custom":
        print(f"\n[tool progress] {item}")


**The gotcha worth remembering:** if you read `stream.messages` all the way to the end *first*, and only then try `stream.extensions["custom"]`, you'll get **nothing** — no error, just silence. Each projection is a single-pass feed over one shared run; anything you don't actively read *while the run is happening* is gone once the run finishes. That's exactly why we read both together with `stream.interleave(...)` above, instead of two separate loops.


## Recap

| What you want to watch | What to use | Step |
|---|---|---|
| The model's reply, as it's typed | `stream.messages` → `message.text` | 1 |
| The model's private thinking, before it answers | `message.reasoning` | 2 |
| The model writing a tool call's arguments | `message.tool_calls` | 3 |
| A tool actually running, and its result | `stream.tool_calls` → `.tool_name`, `.input`, `.output`, `.error` | 4 |
| The whole conversation state, live, step by step | `stream.values` | 5 |
| The final state, once the run is done | `stream.output` | 5 |
| Several of the above together, in one live timeline | `stream.interleave(...)` | 6 |
| The plan, and any re-planning | **Not a separate API** — just more `message.text`, driven by your system prompt | 6 |
| A tool narrating its own progress, live | `get_stream_writer()` inside the tool + a registered `StreamTransformer` → `stream.extensions["custom"]` | 7 |

**One idea to remember:** `stream_events(..., version="v3")` gives you a small set of live feeds. Learn each one on its own first — that's what Steps 1–5 were for — then combine only the ones your app actually needs with `interleave`. And when you do combine feeds, read them **together** (`interleave`, or `asyncio.gather`) — reading one all the way to the end before touching another silently loses whatever that other feed produced.


## Try it yourself

1. In Step 2, ask a question with more to reason about (e.g. "Should I visit Goa in July or December, and why?") and see if `message.reasoning` produces anything for your model.
2. In Step 3, print `chunk` from `message.tool_calls` for a couple of different questions — notice the arguments arrive in small pieces (sometimes even partial JSON) before `.get()` gives you the finished version.
3. In Step 6, also add `stream.values` into the picture: after the run finishes, use `stream.output` (as in Step 5) to print the total number of messages, and compare it to how many `[tool ran]` lines you saw live.
4. In Step 6, lower the budget to INR 15,000 and see whether the agent re-plans **twice**, and confirm that with the tool-call counter at the end (it should now show 5 or more).
5. In Step 7, add `get_stream_writer()` calls into `search_flights` and `estimate_total_cost` too (copy the pattern from `check_weather_verbose`), then rebuild the Step 6 agent using these "verbose" versions and a `transformers=[ToolProgressTransformer]` + `stream.interleave("messages", "tool_calls", "custom")` — watch the full plan-execute-replan run, now with each tool narrating itself.
6. Deliberately reproduce the Step 7 gotcha: read `stream.messages` to completion in one loop, then try `stream.extensions["custom"]` in a second loop right after. Confirm it comes back empty, and explain in your own words why.
